# 高级分析

本教程演示 HIcosmo 的高级功能。

**关键 API**：
- `likelihood1 + likelihood2` — 联合似然（+ 运算符）
- `ILCDM(beta=...)` — 相互作用暗能量模型
- `SH0ESLikelihood()` — SH0ES 距离阶梯
- `LikelihoodDiagnostics()` — 似然诊断
- `information_criteria()` — 模型选择（AIC/BIC）

In [ ]:
import hicosmo as hc
hc.init()

## 1. 多探针联合分析

In [ ]:
from hicosmo.samplers import MCMC
from hicosmo.likelihoods import SN_likelihood, BAO_likelihood, Planck
from hicosmo.models import LCDM

# 创建似然
sne = SN_likelihood(LCDM, "pantheon+")
bao = BAO_likelihood(LCDM, "desi2024")
cmb = Planck(LCDM)

# 联合似然（使用 + 运算符）
joint = sne + bao + cmb

params = {
    'H0': (68.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
}

mcmc = MCMC(params, joint, chain_name='joint_all')
mcmc.run(num_samples=3000)
mcmc.print_summary()

In [ ]:
# 多探针比较
mcmc_sn = MCMC(params, sne, chain_name='adv_sn')
mcmc_sn.run(num_samples=2000)

mcmc_bao = MCMC(params, bao, chain_name='adv_bao')
mcmc_bao.run(num_samples=2000)

from hicosmo.visualization import Plotter
plotter = Plotter(['adv_sn', 'adv_bao', 'joint_all'], labels=['SNe', 'BAO', 'Joint'])
plotter.corner(['H0', 'Omega_m'], filename='figures/09_multiprobe.pdf')

## 2. 模型比较（AIC/BIC）

In [ ]:
from hicosmo.models import wCDM
from hicosmo.visualization import information_criteria

# wCDM 模型
sne_wcdm = SN_likelihood(wCDM, "pantheon+")

params_wcdm = {
    'H0': (70.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
    'w': (-1.0, -2.0, 0.0),
}

mcmc_wcdm = MCMC(params_wcdm, sne_wcdm, chain_name='wcdm_adv')
samples_wcdm = mcmc_wcdm.run(num_samples=2000)
mcmc_wcdm.print_summary()

In [ ]:
# 计算信息准则
ic = information_criteria(
    samples=samples_wcdm,
    log_likelihood_fn=sne_wcdm,
    num_data=sne_wcdm.n_sne,
    param_names=['H0', 'Omega_m', 'w']
)

print(f"χ²_min = {ic['chi2_min']:.2f}")
print(f"AIC = {ic['aic']:.2f}")
print(f"BIC = {ic['bic']:.2f}")

## 3. 相互作用暗能量（ILCDM）

In [ ]:
from hicosmo.models import ILCDM
import numpy as np
import matplotlib.pyplot as plt

# 不同 β 值
z = np.linspace(0, 3, 100)
for beta in [-0.05, 0.0, 0.05]:
    model = ILCDM(H0=67.36, Omega_m=0.3153, beta=beta)
    E_z = [model.E_z(zi) for zi in z]
    plt.plot(z, E_z, label=f'β = {beta}')

plt.xlabel('z'); plt.ylabel('E(z)'); plt.legend()
plt.title('ILCDM: 相互作用强度对比')
plt.savefig('figures/09_ilcdm.pdf', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ILCDM Fisher 预测
from hicosmo.fisher import IntensityMappingFisher

ilcdm = ILCDM(H0=67.36, Omega_m=0.3153, beta=0.001)
result = IntensityMappingFisher.forecast(
    survey='ska1_mid_band2',
    cosmology=ilcdm,
    params=['beta', 'H0', 'Omega_m']
)
print(result)

## 4. SH0ES 距离阶梯

In [ ]:
from hicosmo.likelihoods import SH0ESLikelihood

# SH0ES + SNe 联合
shoes = SH0ESLikelihood(LCDM)
joint_shoes = sne + shoes

params_shoes = {
    'H0': (72.0, 65.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
}

mcmc_shoes = MCMC(params_shoes, joint_shoes, chain_name='sne_shoes')
mcmc_shoes.run(num_samples=2000)
mcmc_shoes.print_summary()

## 5. 似然诊断

In [ ]:
from hicosmo.samplers import LikelihoodDiagnostics

# 诊断似然函数
diag = LikelihoodDiagnostics(sne, params)
result = diag.run(n_tests=50)
diag.print_report(result)

## 6. 高级 MCMC 配置

In [ ]:
# 优化初始化
mcmc_opt = MCMC(
    params, sne,
    chain_name='optimized',
    optimize_init=True,
    max_opt_iterations=500
)

# 采样器选择
mcmc_emcee = MCMC(params, sne, chain_name='emcee_test', sampler='emcee')
mcmc_nuts = MCMC(params, sne, chain_name='nuts_test', sampler='numpyro')

print("采样器: 'numpyro' (NUTS, 梯度加速) 或 'emcee' (无需梯度)")

## 7. 派生参数

In [ ]:
# 模型派生参数
lcdm = LCDM(H0=67.36, Omega_m=0.3153, Omega_b=0.0493)

rd = lcdm.sound_horizon_drag()
H0_rd = lcdm.params['H0'] * rd / 100

print(f"r_d = {rd:.2f} Mpc")
print(f"H0 × r_d = {H0_rd:.2f} km/s")

## API 速查

```python
from hicosmo.samplers import MCMC, LikelihoodDiagnostics
from hicosmo.likelihoods import SN_likelihood, BAO_likelihood, Planck, SH0ESLikelihood
from hicosmo.models import LCDM, wCDM, ILCDM
from hicosmo.visualization import information_criteria

# === 联合似然 ===
joint = sne + bao + cmb  # + 运算符组合

# === 模型比较 ===
ic = information_criteria(
    samples=samples,
    log_likelihood_fn=likelihood,
    num_data=N,
    param_names=['H0', 'Omega_m']
)
print(f"AIC={ic['aic']}, BIC={ic['bic']}")

# === ILCDM ===
ilcdm = ILCDM(H0=67.36, Omega_m=0.3153, beta=0.01)  # β: 相互作用强度

# === SH0ES ===
shoes = SH0ESLikelihood(LCDM)
joint = sne + shoes

# === 诊断 ===
diag = LikelihoodDiagnostics(likelihood, params)
diag.run(n_tests=50)

# === 高级 MCMC ===
mcmc = MCMC(
    params, likelihood,
    sampler='numpyro',     # 或 'emcee'
    optimize_init=True     # 优化初始点
)

# === 派生参数 ===
rd = model.sound_horizon_drag()  # 声学视界 [Mpc]
```

### 模型层次

| 模型 | 参数 | 说明 |
|------|------|------|
| LCDM | H0, Ω_m | 标准模型 |
| wCDM | + w | 常数暗能量 |
| CPL | + w0, wa | 演化暗能量 |
| ILCDM | + β | 相互作用暗能量 |